# 05 — Posterior analysis (pipeline)

Maps, **ranking divergence**, and **case-study** posterior distributions for tract-level job accessibility (BYM2 output from notebook **04**).

## Inputs

- **Posterior summary:** `data/processed/posteriors/<run_id>_posterior_summary.parquet` — pin **`fit_raw_zscore_x`** (raw **X**, primary) vs **`fit_spatial_plus_x`** (**Spatial+** sensitivity) in §1a, or use `PIPELINE_POSTERIOR_STEM` / newest file when both pins are off. Constants live in `src.utils.config`.
- **Optional full draws:** `<run_id>_idata.nc` — enables **Fig 6** fan charts of tract-level `mu` (without loading the huge long samples parquet).
- **Deterministic baseline:** latest `tract_accessibility_bundle__*.parquet` (same `jobs_C000_*` column as nb04’s primary threshold).
- **ACS:** latest `artifacts/tables/eda/eda__acs_sd_tract_attributes__*.csv` for `disadvantage_z` and labels.
- **Geometries:** TIGER tract shapefile (same resolution as nb04).

## Outputs (paper-aligned)

| Target | Artifact |
|--------|----------|
| Fig 3 — Posterior mean jobs | `artifacts/figures/pipeline__05_posterior_mean_jobs__<run_id>.png` |
| Fig 4 — Low-access probability | `artifacts/figures/pipeline__05_p_low_access_q25__<run_id>.png` |
| Fig 5 — 95% CI width (log1p) | `artifacts/figures/pipeline__05_ci_width_log1p__<run_id>.png` |
| Fig 6 — Case-study posteriors | `artifacts/figures/pipeline__05_case_study_mu__<run_id>.png` |
| Fig 7 — Rank divergence | `artifacts/figures/pipeline__05_rank_divergence__<run_id>.png` |
| Fig 6b — Fan chart (KDE on jobs scale) | `pipeline__05_case_study_fan_kde_jobs__<run_id>.png` (run §7 after §5) |
| Fig 8 — Wasserstein vs reference pool (D003) | `pipeline__05_wasserstein_map__<run_id>.png`, `pipeline__05_wasserstein_tract__<run_id>.csv` (§8) |
| Multi-threshold (30/45/60) | `pipeline__05_multithreshold_equity__<run_id>.csv`, `pipeline__05_exceedance_shift_by_threshold__<run_id>.csv` (§9) |
| Hook / narrative | `pipeline__05_uncertainty_hook_candidates__<run_id>.csv`, `pipeline__05_hook_near_q25_exceedance_ambiguous__<run_id>.csv` (§10) |

**Naming note:** In nb04 exports, `exceedance_prob_45min` is **P(μ below city Q25 on log1p jobs | draws)** — a “low structural accessibility” tail probability, not an exceedance above a high bar.

**Narrative:** Deterministic vs Bayesian **ranks** are typically **near-identical** (ρ ≈ 1). The paper **hook** pivots to **uncertainty** and **threshold / tail probability** — tracts near the Q25 cut with **wide** credible intervals or **ambiguous** exceedance (hook cell in section 10).

**Previous:** [`04_bayesian_model.ipynb`](04_bayesian_model.ipynb) · **Next:** [`06_divergence_metrics.ipynb`](06_divergence_metrics.ipynb)



In [1]:
from __future__ import annotations

import os
import sys
from pathlib import Path

import numpy as np
import pandas as pd

REPO_ROOT = next(
    (d for d in [Path.cwd().resolve(), *Path.cwd().resolve().parents] if (d / "configs" / "san_diego.yaml").exists()),
    None,
)
if REPO_ROOT is None:
    raise FileNotFoundError("Could not find configs/san_diego.yaml.")
sys.path.insert(0, str(REPO_ROOT))

import importlib  # noqa: E402

# Clear any cached imports that may point at a stale file.
sys.modules.pop("src.utils.config", None)
sys.modules.pop("src.utils", None)
sys.modules.pop("src", None)

import src.utils.config as cfg  # noqa: E402

# In notebooks, modules can stay cached across runs (especially after edits).
# Reloading makes this cell robust to that state.
cfg = importlib.reload(cfg)

RUN_FIT_RAW_ZSCORE_X = cfg.RUN_FIT_RAW_ZSCORE_X
RUN_FIT_SPATIAL_PLUS_X = cfg.RUN_FIT_SPATIAL_PLUS_X
load_merged_config = cfg.load_merged_config

CONFIG = load_merged_config(REPO_ROOT)
ACC_CFG = CONFIG.get("accessibility", {})
T_PRIMARY = int(ACC_CFG.get("travel_time_threshold_min", 45))
JOB_COL = f"jobs_C000_{T_PRIMARY}min"

POSTERIOR_DIR = REPO_ROOT / "data" / "processed" / "posteriors"
ART_TAB_EDA = REPO_ROOT / "artifacts" / "tables" / "eda"
ART_TAB_PIPE = REPO_ROOT / "artifacts" / "tables" / "pipeline"
ART_FIG = REPO_ROOT / "artifacts" / "figures"
ART_TAB_EDA.mkdir(parents=True, exist_ok=True)
ART_TAB_PIPE.mkdir(parents=True, exist_ok=True)
ART_FIG.mkdir(parents=True, exist_ok=True)

_LOCKED_RID: str | None = None


def _apply_posterior_pin(rid: str) -> None:
    # Set RID / sum_path / idata_path; raise if a different run was already pinned this session.
    global RID, sum_path, idata_path, _LOCKED_RID
    if _LOCKED_RID is not None and _LOCKED_RID != rid:
        raise RuntimeError(
            f"Posterior run already pinned to {_LOCKED_RID!r}. Restart the kernel, then enable only one of: "
            "raw-z-score-X pin, Spatial+ pin, or leave both off and use the default resolver."
        )
    _LOCKED_RID = rid
    RID = rid
    sum_path = POSTERIOR_DIR / f"{RID}_posterior_summary.parquet"
    idata_path = POSTERIOR_DIR / f"{RID}_idata.nc"
    if not sum_path.is_file():
        raise FileNotFoundError(
            f"No posterior summary for run {rid!r}: {sum_path}\n"
            "Fit it with notebooks/04_bayesian_model.ipynb (set PIPELINE_RUN_ID=fit_raw_zscore_x or fit_spatial_plus_x)."
        )

bundles = sorted((REPO_ROOT / "data" / "processed" / "accessibility").glob("tract_accessibility_bundle__*.parquet"))
if not bundles:
    raise FileNotFoundError("No tract_accessibility_bundle__*.parquet — run notebook 03 first.")
bundle_path = bundles[-1]

acs_files = sorted(ART_TAB_EDA.glob("eda__acs_sd_tract_attributes__*.csv"))
if not acs_files:
    raise FileNotFoundError("No eda__acs_sd_tract_attributes__*.csv — run EDA 03 first.")
acs_path = acs_files[-1]

census_cfg = CONFIG.get("census", {})
state_fips = str(census_cfg.get("state_fips", CONFIG.get("state_fips", "06"))).zfill(2)
county_fips = str(census_cfg.get("county_fips", CONFIG.get("county_fips", "073"))).zfill(3)
acs_year = int(census_cfg.get("acs_year", 2023))
tiger_dir = REPO_ROOT / "data" / "raw" / "census" / f"tl_{acs_year}_{state_fips}_tract"
tiger_shp = tiger_dir / f"tl_{acs_year}_{state_fips}_tract.shp"
if not tiger_shp.is_file():
    raise FileNotFoundError(f"Missing TIGER: {tiger_shp}")

print("REPO_ROOT =", REPO_ROOT)
print("Bundle:", bundle_path.relative_to(REPO_ROOT))
print("T_PRIMARY =", T_PRIMARY, "→", JOB_COL)



REPO_ROOT = C:\Users\sardo\OneDrive\Desktop\Classes\Projects\BayesTransitEquity
Bundle: data\processed\accessibility\tract_accessibility_bundle__2026-04-03.parquet
T_PRIMARY = 45 → jobs_C000_45min


### 1a. Posterior run (`RID`) — which BYM2 fit?

**`run_id`** is a short slug in filenames (not a calendar date). Two presets match notebook **04**:

| `run_id` | Meaning |
|----------|---------|
| **`fit_raw_zscore_x`** | BYM2 on **z-scored raw covariates** (`PIPELINE_NO_SPATIAL_PLUS=1` when fitting) — **paper primary** (D011). |
| **`fit_spatial_plus_x`** | BYM2 on **Spatial+** residualized **X** (eigen removal before ICAR) — **sensitivity** estimand. |

| Goal | What to run |
|------|-------------|
| **Raw X only (default)** | **`SELECT_FIT_RAW_ZSCORE_X = True`**, **`SELECT_FIT_SPATIAL_PLUS_X = False`**, then **Run All**. |
| **Spatial+ only** | **`SELECT_FIT_SPATIAL_PLUS_X = True`**, **`SELECT_FIT_RAW_ZSCORE_X = False`**, then **Run All**. |
| **Neither pin** | Both **`False`** → run the **default resolver** cell (`PIPELINE_POSTERIOR_STEM` env, else **newest** `*_posterior_summary.parquet`). |

**Do not** set both select flags **`True`** — the second pin raises (restart kernel).

**CLI:** `PIPELINE_POSTERIOR_STEM=fit_raw_zscore_x` (or `fit_spatial_plus_x`) when both pins are off.

**Migrate old files:** `python scripts/migrate_bym2_run_ids.py` renames legacy `2026-04-05` / `2026-04-06` stems to the slugs above.




In [2]:
# ── Pin: **fit_raw_zscore_x** — posterior from BYM2 on raw z-scored X (primary) ──
SELECT_FIT_RAW_ZSCORE_X = True
if SELECT_FIT_RAW_ZSCORE_X:
    _apply_posterior_pin(RUN_FIT_RAW_ZSCORE_X)
    print("  idata.nc exists:", idata_path.is_file())



  idata.nc exists: True


In [3]:
# ── Pin: **fit_spatial_plus_x** — posterior from BYM2 on Spatial+ X (sensitivity) ──
SELECT_FIT_SPATIAL_PLUS_X = False
if SELECT_FIT_SPATIAL_PLUS_X:
    _apply_posterior_pin(RUN_FIT_SPATIAL_PLUS_X)
    print("  idata.nc exists:", idata_path.is_file())



In [4]:
# ── Default resolver (env / newest summary) ───────────────────────────────────
# Runs when neither pin cell above called _apply_posterior_pin. Priority: PIPELINE_POSTERIOR_STEM,
# then newest *_posterior_summary.parquet by mtime.
if _LOCKED_RID is None:
    stem_env = os.environ.get("PIPELINE_POSTERIOR_STEM", "").strip()
    candidates = sorted(POSTERIOR_DIR.glob("*_posterior_summary.parquet"), key=lambda p: p.stat().st_mtime)
    if not candidates:
        raise FileNotFoundError("No *_posterior_summary.parquet — run notebook 04 first.")
    if stem_env:
        _apply_posterior_pin(stem_env)
    else:
        latest = candidates[-1]
        _apply_posterior_pin(latest.stem.replace("_posterior_summary", ""))
else:
    print("Posterior run already pinned:", _LOCKED_RID)

print("Posterior summary:", sum_path.relative_to(REPO_ROOT))
print("RID =", RID)
print("idata.nc exists:", idata_path.is_file())



Posterior run already pinned: fit_raw_zscore_x
Posterior summary: data\processed\posteriors\fit_raw_zscore_x_posterior_summary.parquet
RID = fit_raw_zscore_x
idata.nc exists: True


In [5]:
import geopandas as gpd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display
from scipy import stats

summary = pd.read_parquet(sum_path)
summary["GEOID"] = summary["GEOID"].astype(str).str.zfill(11)

acc = pd.read_parquet(bundle_path)
acc["GEOID"] = acc["GEOID"].astype(str).str.zfill(11)
_job_mt = [f"jobs_C000_{t}min" for t in (30, 45, 60)]
acc_cols = ["GEOID"] + [c for c in _job_mt if c in acc.columns]
if JOB_COL not in acc_cols:
    acc_cols.append(JOB_COL)
acc = acc[[c for c in acc_cols if c in acc.columns]]

acs = pd.read_csv(acs_path)
acs["GEOID"] = acs["GEOID"].astype(str).str.zfill(11)
acs_keep = ["GEOID", "disadvantage_z", "B01003_001E", "poverty_rate"]
acs_keep = [c for c in acs_keep if c in acs.columns]
acs = acs[acs_keep]

merged = summary.merge(acc, on="GEOID", how="inner", validate="one_to_one").merge(acs, on="GEOID", how="left")
if len(merged) < len(summary):
    print("warn: bundle join dropped", len(summary) - len(merged), "tracts vs summary")

# Match nb04: y_mean / y_sd on log1p(jobs) for this cohort (for back-transforming mu draws)
y_raw = pd.to_numeric(merged[JOB_COL], errors="coerce").fillna(0.0).to_numpy()
y_log = np.log1p(y_raw)
y_mean = float(np.mean(y_log))
y_sd = float(np.std(y_log, ddof=1)) or 1.0

exc_col = f"exceedance_prob_{T_PRIMARY}min"
if exc_col not in merged.columns:
    raise KeyError(f"Expected column {exc_col!r} in posterior summary — re-run nb04.")

merged["ci_width_log1p"] = merged["ci_upper_95_log1p"] - merged["ci_lower_95_log1p"]
merged["rank_det"] = merged[JOB_COL].rank(ascending=False, method="average")
merged["rank_bayes"] = merged["posterior_mean_jobs"].rank(ascending=False, method="average")

tracts_all = gpd.read_file(tiger_shp)
tracts_all["GEOID"] = tracts_all["GEOID"].astype(str).str.zfill(11)
tracts_sd = tracts_all[tracts_all["COUNTYFP"] == county_fips].copy()
map_gdf = tracts_sd.merge(merged, on="GEOID", how="left")

display(Markdown(f"**Tracts in posterior summary:** {len(merged)}  |  **y_mean, y_sd** (log1p cohort): {y_mean:.4f}, {y_sd:.4f}"))
merged[[ "GEOID", "posterior_mean_jobs", JOB_COL, exc_col, "disadvantage_z"]].head()



**Tracts in posterior summary:** 720  |  **y_mean, y_sd** (log1p cohort): 9.0656, 1.2501

,GEOID,posterior_mean_jobs,jobs_C000_45min,exceedance_prob_45min,disadvantage_z
0,06073000100,48968.620382,49803.0,0.0,-1.217822
1,06073000201,32603.042229,32749.0,0.0,0.681944
2,06073000202,58406.714839,59221.0,0.0,-0.182962
3,06073000301,46732.727973,47011.0,0.0,1.006753
4,06073000302,84832.765947,85813.0,0.0,0.364107


In [6]:
# Fig 3 — Posterior mean jobs (tract choropleth)
fig, ax = plt.subplots(figsize=(9, 9))
map_gdf.plot(
    column="posterior_mean_jobs",
    cmap="YlOrRd",
    legend=True,
    ax=ax,
    missing_kwds={"color": "lightgrey"},
    legend_kwds={"label": "Posterior mean jobs"},
)
ax.set_axis_off()
ax.set_title(f"Posterior mean jobs reachable ≤ {T_PRIMARY} min (BYM2, run {RID})")
plt.tight_layout()
p = ART_FIG / f"pipeline__05_posterior_mean_jobs__{RID}.png"
fig.savefig(p, dpi=200, bbox_inches="tight")
plt.close(fig)
display(Markdown(f"Saved `{p.relative_to(REPO_ROOT)}`"))



Saved `artifacts\figures\pipeline__05_posterior_mean_jobs__fit_raw_zscore_x.png`

In [7]:
# Fig 4 — P(μ below Q25 | posterior) — "low access" tail probability
fig, ax = plt.subplots(figsize=(9, 9))
map_gdf.plot(
    column=exc_col,
    cmap="magma_r",
    vmin=0,
    vmax=1,
    legend=True,
    ax=ax,
    missing_kwds={"color": "lightgrey"},
    legend_kwds={"label": "Probability"},
)
ax.set_axis_off()
ax.set_title(
    f"P(model μ below city Q25 on log1p jobs) — threshold {T_PRIMARY} min (run {RID})"
)
plt.tight_layout()
p = ART_FIG / f"pipeline__05_p_low_access_q25__{RID}.png"
fig.savefig(p, dpi=200, bbox_inches="tight")
plt.close(fig)
display(Markdown(f"Saved `{p.relative_to(REPO_ROOT)}`"))



Saved `artifacts\figures\pipeline__05_p_low_access_q25__fit_raw_zscore_x.png`

In [8]:
# Fig 5 — Posterior uncertainty: 95% CI width on log1p scale
fig, ax = plt.subplots(figsize=(9, 9))
map_gdf.plot(
    column="ci_width_log1p",
    cmap="viridis",
    legend=True,
    ax=ax,
    missing_kwds={"color": "lightgrey"},
    legend_kwds={"label": "log1p jobs, 95% CI width"},
)
ax.set_axis_off()
ax.set_title(f"Posterior 95% CI width (log1p jobs), {T_PRIMARY} min (run {RID})")
plt.tight_layout()
p = ART_FIG / f"pipeline__05_ci_width_log1p__{RID}.png"
fig.savefig(p, dpi=200, bbox_inches="tight")
plt.close(fig)
display(Markdown(f"Saved `{p.relative_to(REPO_ROOT)}`"))



Saved `artifacts\figures\pipeline__05_ci_width_log1p__fit_raw_zscore_x.png`

## 4. Ranking divergence (Fig 7)

**Spearman** and **Kendall** correlation between deterministic and Bayesian **ranks** (1 = highest jobs). Export full tract table sorted by |Δrank| for appendix / case picking.



In [9]:
# Fig 7 — Deterministic vs Bayesian ranks (higher jobs → rank 1)
m = merged.dropna(subset=[JOB_COL, "posterior_mean_jobs"]).copy()
m["rank_det"] = m[JOB_COL].rank(ascending=False, method="average")
m["rank_bayes"] = m["posterior_mean_jobs"].rank(ascending=False, method="average")
m["rank_delta"] = m["rank_bayes"] - m["rank_det"]

rho_s, p_s = stats.spearmanr(m["rank_det"], m["rank_bayes"])
tau_k, p_k = stats.kendalltau(m["rank_det"], m["rank_bayes"])

fig, ax = plt.subplots(figsize=(6.5, 6.5))
hb = ax.hexbin(m["rank_det"], m["rank_bayes"], gridsize=35, cmap="Blues", mincnt=1)
plt.colorbar(hb, ax=ax, label="tract count")
ax.plot([1, len(m)], [1, len(m)], "k--", lw=1, alpha=0.6, label="y = x")
ax.set_xlabel(f"Deterministic rank ({JOB_COL}, 1 = most jobs)")
ax.set_ylabel("Bayesian rank (posterior mean jobs, 1 = highest)")
ax.set_title(f"Rank divergence (Spearman ρ = {rho_s:.3f}, Kendall τ = {tau_k:.3f})")
ax.legend(loc="lower right")
plt.tight_layout()
pfig = ART_FIG / f"pipeline__05_rank_divergence__{RID}.png"
fig.savefig(pfig, dpi=200, bbox_inches="tight")
plt.close(fig)

rank_path = ART_TAB_PIPE / f"pipeline__05_rank_divergence__{RID}.csv"
(
    m[
        [
            "GEOID",
            "rank_det",
            "rank_bayes",
            "rank_delta",
            JOB_COL,
            "posterior_mean_jobs",
            "disadvantage_z",
            exc_col,
            "ci_width_log1p",
        ]
    ]
    .assign(_abs_delta=lambda d: d["rank_delta"].abs())
    .sort_values("_abs_delta", ascending=False)
    .drop(columns=["_abs_delta"])
    .to_csv(rank_path, index=False)
)

display(Markdown(f"Saved `{pfig.relative_to(REPO_ROOT)}` and `{rank_path.relative_to(REPO_ROOT)}`"))
display(Markdown(f"**Rank correlation:** Spearman ρ = {rho_s:.4f} (*p* = {p_s:.2e}), Kendall τ = {tau_k:.4f} (*p* = {p_k:.2e})"))



Saved `artifacts\figures\pipeline__05_rank_divergence__fit_raw_zscore_x.png` and `artifacts\tables\pipeline\pipeline__05_rank_divergence__fit_raw_zscore_x.csv`

**Rank correlation:** Spearman ρ = 1.0000 (*p* = 0.00e+00), Kendall τ = 0.9973 (*p* = 0.00e+00)

## 5. Case-study μ posteriors (Fig 6)

Uses **`{RID}_idata.nc`** only (avoids the multi-GB long-format parquet). Four exemplar tracts: largest Bayesian upgrade/downgrade vs deterministic ordering, and high/low **disadvantage_z** contrast pairs aligned with **F020 / F024**.



In [10]:
# Fig 6 — Case studies: posterior of μ on log1p scale (needs idata.nc)
import matplotlib.pyplot as plt

# 1) Largest upward Bayesian reassessment (Bayesian rank better than deterministic = lower rank number)
labels = {}
m = merged.dropna(subset=[JOB_COL, "posterior_mean_jobs", "disadvantage_z"]).copy()
m["rank_det"] = m[JOB_COL].rank(ascending=False, method="average")
m["rank_bayes"] = m["posterior_mean_jobs"].rank(ascending=False, method="average")
m["rank_delta"] = m["rank_bayes"] - m["rank_det"]
if not m.empty:
    i = m["rank_delta"].idxmin()
    g = str(m.loc[i, "GEOID"])
    labels[g] = "largest Bayesian upgrade vs deterministic rank"

# 2) Largest downward reassessment
if not m.empty:
    i = m["rank_delta"].idxmax()
    g = str(m.loc[i, "GEOID"])
    labels[g] = "largest Bayesian downgrade vs deterministic rank"

# 3) High disadvantage, high posterior mean (urban core story)
hd = m["disadvantage_z"] >= m["disadvantage_z"].quantile(0.9)
if hd.any():
    sub = m.loc[hd].nlargest(1, "posterior_mean_jobs")
    g = str(sub.iloc[0]["GEOID"])
    labels[g] = "high disadvantage, high posterior mean"

# 4) Low disadvantage, low posterior mean
ld = m["disadvantage_z"] <= m["disadvantage_z"].quantile(0.1)
if ld.any():
    sub = m.loc[ld].nsmallest(1, "posterior_mean_jobs")
    g = str(sub.iloc[0]["GEOID"])
    labels[g] = "low disadvantage, low posterior mean"

case_geoids = list(dict.fromkeys(labels.keys()))
case_rows = [{"GEOID": g, "label": labels[g]} for g in case_geoids]

if not case_geoids:
    display(Markdown("_No case-study GEOIDs selected — check merged tract counts and `disadvantage_z` coverage._"))
elif not idata_path.is_file():
    display(Markdown("_Skip Fig 6: `idata.nc` not found for this run — re-run nb04 with NetCDF export enabled._"))
else:  # case_geoids non-empty and idata present
    import arviz as az

    idata = az.from_netcdf(idata_path)
    mu = idata.posterior["mu"].values
    tract_coord = [
        str(x).zfill(11) for x in np.asarray(idata.posterior["mu"].coords["tract"].values).ravel()
    ]
    fig, axes = plt.subplots(2, 2, figsize=(9, 7), sharex=False, sharey=False)
    axes = axes.ravel()
    for ax, g in zip(axes, case_geoids[:4]):
        if g not in tract_coord:
            ax.set_visible(False)
            continue
        k = tract_coord.index(g)
        draws_log1p = (mu[:, :, k].ravel() * y_sd) + y_mean
        ax.hist(draws_log1p, bins=60, density=True, color="steelblue", alpha=0.75)
        ax.axvline(float(merged.loc[merged["GEOID"] == g, "posterior_mean_log1p"].iloc[0]), color="red", ls="--", lw=1)
        ax.set_title(f"{g}\\n{labels.get(g, '')}", fontsize=8)
        ax.set_xlabel("μ (log1p jobs)")
    for j in range(len(case_geoids[:4]), 4):
        axes[j].set_visible(False)
    plt.suptitle(f"Tract-level μ posterior (log1p jobs), run {RID}")
    plt.tight_layout()
    p_cs = ART_FIG / f"pipeline__05_case_study_mu__{RID}.png"
    fig.savefig(p_cs, dpi=200, bbox_inches="tight")
    plt.close(fig)
    display(Markdown(f"Saved `{p_cs.relative_to(REPO_ROOT)}`"))

case_manifest = ART_TAB_PIPE / f"pipeline__05_case_study_manifest__{RID}.csv"
pd.DataFrame(case_rows).to_csv(case_manifest, index=False)
display(Markdown(f"Case list → `{case_manifest.relative_to(REPO_ROOT)}`"))



Saved `artifacts\figures\pipeline__05_case_study_mu__fit_raw_zscore_x.png`

Case list → `artifacts\tables\pipeline\pipeline__05_case_study_manifest__fit_raw_zscore_x.csv`

## 6. Summary CSV

Spearman(**variable**, **disadvantage_z**) for key posterior-derived quantities plus rank–rank association.



In [11]:
# Summary table (equity + rank stability)
rows = []
dz = merged["disadvantage_z"]
for name, col in [
    ("posterior_mean_jobs", "posterior_mean_jobs"),
    ("posterior_sd_log1p", "posterior_sd_log1p"),
    (exc_col, exc_col),
    ("ci_width_log1p", "ci_width_log1p"),
]:
    pair = pd.DataFrame({"z": dz, "v": merged[col]}).dropna()
    if len(pair) > 2:
        r, p = stats.spearmanr(pair["z"], pair["v"])
        rows.append({"variable": name, "spearman_vs_disadvantage_z": r, "p_value": p})

m2 = merged.dropna(subset=[JOB_COL, "posterior_mean_jobs", "rank_det", "rank_bayes"])
rho_s, p_s = stats.spearmanr(m2["rank_det"], m2["rank_bayes"])
rows.append({"variable": "rank_det_vs_rank_bayes", "spearman_vs_disadvantage_z": rho_s, "p_value": p_s})

out = pd.DataFrame(rows)
summary_csv = ART_TAB_PIPE / f"pipeline__05_summary__{RID}.csv"
out.to_csv(summary_csv, index=False)
display(Markdown(f"Saved `{summary_csv.relative_to(REPO_ROOT)}`"))
out



Saved `artifacts\tables\pipeline\pipeline__05_summary__fit_raw_zscore_x.csv`

,variable,spearman_vs_disadvantage_z,p_value
0,posterior_mean_jobs,0.469872,8.047024e-41
1,posterior_sd_log1p,-0.044890,2.289625e-01
2,exceedance_prob_45min,-0.466943,2.860300e-40
3,ci_width_log1p,-0.058976,1.138533e-01
4,rank_det_vs_rank_bayes,0.999976,0.000000e+00


## 7. Fig 6b — Fan chart (overlapping KDE on jobs scale)

Posterior **μ** draws back-transformed with **`np.expm1(μ · y_sd + y_mean)`** so the *x*-axis is interpretable job counts. Same case-study GEOIDs as section 5.



In [12]:
# Fig 6b — Overlapping KDE fan chart (posterior μ → jobs scale)
from scipy.stats import gaussian_kde

if "case_geoids" not in dir() or not case_geoids or not idata_path.is_file():
    display(Markdown("_Skip fan chart: run case-study cell first and ensure `idata.nc` exists._"))
else:
    import arviz as az

    idata_fc = az.from_netcdf(idata_path)
    mu_fc = idata_fc.posterior["mu"].values
    tract_coord_fc = [
        str(x).zfill(11) for x in np.asarray(idata_fc.posterior["mu"].coords["tract"].values).ravel()
    ]
    fig, ax = plt.subplots(figsize=(9, 5))
    cmap = plt.cm.tab10(np.linspace(0, 0.9, max(1, min(4, len(case_geoids)))))
    for idx, g in enumerate(case_geoids[:4]):
        if g not in tract_coord_fc:
            continue
        k = tract_coord_fc.index(g)
        draws_jobs = np.expm1((mu_fc[:, :, k].ravel() * y_sd) + y_mean)
        draws_jobs = draws_jobs[np.isfinite(draws_jobs) & (draws_jobs >= 0)]
        if np.std(draws_jobs) < 1e-9:
            continue
        kde = gaussian_kde(draws_jobs)
        lo, hi = np.percentile(draws_jobs, [1, 99])
        xs = np.linspace(lo, hi, 256)
        lbl = f"{g}"
        if "labels" in dir() and g in labels:
            lbl += f" — {str(labels[g])[:34]}"
        ax.plot(xs, kde(xs), color=cmap[idx], lw=2, label=lbl)
    ax.set_xlabel("Posterior μ implied jobs (expm1 scale)")
    ax.set_ylabel("Density")
    ax.set_title(f"Case-study μ posteriors (jobs), KDE overlay — run {RID}")
    ax.legend(fontsize=7, loc="upper right")
    plt.tight_layout()
    pfan = ART_FIG / f"pipeline__05_case_study_fan_kde_jobs__{RID}.png"
    fig.savefig(pfan, dpi=200, bbox_inches="tight")
    plt.close(fig)
    display(Markdown(f"Saved `{pfan.relative_to(REPO_ROOT)}`"))



Saved `artifacts\figures\pipeline__05_case_study_fan_kde_jobs__fit_raw_zscore_x.png`

## 8. Fig 8 — Wasserstein distance to a well-served reference pool (**D003**)

For each tract, **1D Wasserstein** distance between its posterior jobs samples and a **reference pool** = all posterior draws from tracts in the **top quartile** of **posterior_mean_jobs** (city-relative “well served”). High values = distributional shift vs that reference.



In [13]:
# Fig 8 — Wasserstein distance vs top-quartile “well served” pool (D003)
from scipy.stats import wasserstein_distance

if not idata_path.is_file():
    display(Markdown("_Skip Wasserstein: `idata.nc` missing._"))
else:
    import arviz as az

    idata_w = az.from_netcdf(idata_path)
    mu_w = idata_w.posterior["mu"].values
    Cw, Dw, Nmu = mu_w.shape
    jobs_samps = np.expm1(mu_w * y_sd + y_mean)
    flat = jobs_samps.reshape(Cw * Dw, Nmu)
    tract_coord_w = [
        str(x).zfill(11) for x in np.asarray(idata_w.posterior["mu"].coords["tract"].values).ravel()
    ]
    pm_aligned = merged.set_index("GEOID").reindex(tract_coord_w)["posterior_mean_jobs"].to_numpy()
    q75 = float(np.nanquantile(pm_aligned, 0.75))
    top_ix = np.where(pm_aligned >= q75)[0]
    ref_pool = flat[:, top_ix].ravel()
    ref_pool = ref_pool[np.isfinite(ref_pool)]
    ws: list[float] = []
    for i in range(Nmu):
        s = flat[:, i].ravel()
        s = s[np.isfinite(s)]
        if len(s) < 50:
            ws.append(float("nan"))
        else:
            ws.append(float(wasserstein_distance(s, ref_pool)))
    wdf = pd.DataFrame({"GEOID": tract_coord_w, "wasserstein_topq_reference_pool": ws})
    wcsv = ART_TAB_PIPE / f"pipeline__05_wasserstein_tract__{RID}.csv"
    wdf.to_csv(wcsv, index=False)
    _wdrop = [c for c in ("wasserstein_topq_reference_pool",) if c in map_gdf.columns]
    map_w = map_gdf.drop(columns=_wdrop, errors="ignore").merge(wdf, on="GEOID", how="left")
    fig, ax = plt.subplots(figsize=(9, 9))
    map_w.plot(
        column="wasserstein_topq_reference_pool",
        cmap="YlOrRd",
        legend=True,
        ax=ax,
        missing_kwds={"color": "lightgrey"},
        legend_kwds={"label": "Wasserstein₁ (jobs)"},
    )
    ax.set_axis_off()
    ax.set_title(f"Wasserstein₁ vs top-quartile posterior-mean pool — {RID}")
    plt.tight_layout()
    pw = ART_FIG / f"pipeline__05_wasserstein_map__{RID}.png"
    fig.savefig(pw, dpi=200, bbox_inches="tight")
    plt.close(fig)
    display(Markdown(f"Saved `{wcsv.relative_to(REPO_ROOT)}` and `{pw.relative_to(REPO_ROOT)}`"))



Saved `artifacts\tables\pipeline\pipeline__05_wasserstein_tract__fit_raw_zscore_x.csv` and `artifacts\figures\pipeline__05_wasserstein_map__fit_raw_zscore_x.png`

## 9. Multi-threshold sensitivity (30 / 45 / 60 min)

Deterministic **Spearman(jobs, disadvantage_z)** at each OD threshold; **Spearman(exceedance_prob_*, disadvantage_z)** from the **single-threshold** BYM2 fit (μ is trained on primary **T_PRIMARY** only — exceedance columns are sensitivity cuts, see nb04 footer).



In [14]:
# Multi-threshold equity + exceedance vs disadvantage
rows_mt: list[dict] = []
dzm = merged["disadvantage_z"]
for t in (30, 45, 60):
    jc = f"jobs_C000_{t}min"
    if jc not in merged.columns:
        continue
    pair = pd.DataFrame({"z": dzm, "j": pd.to_numeric(merged[jc], errors="coerce")}).dropna()
    if len(pair) > 2:
        r, p = stats.spearmanr(pair["z"], pair["j"])
        rows_mt.append(
            {"threshold_min": t, "metric": "deterministic_jobs_vs_disadvantage_z", "spearman_rho": r, "p_value": p}
        )
for t, exc_c in [(30, "exceedance_prob_30min"), (45, "exceedance_prob_45min"), (60, "exceedance_prob_60min")]:
    if exc_c not in merged.columns:
        continue
    pair = pd.DataFrame({"z": dzm, "e": merged[exc_c]}).dropna()
    if len(pair) > 2:
        r, p = stats.spearmanr(pair["z"], pair["e"])
        rows_mt.append(
            {"threshold_min": t, "metric": "posterior_exceedance_vs_disadvantage_z", "spearman_rho": r, "p_value": p}
        )
mt_df = pd.DataFrame(rows_mt)
mt_path = ART_TAB_PIPE / f"pipeline__05_multithreshold_equity__{RID}.csv"
mt_df.to_csv(mt_path, index=False)
display(Markdown(f"Saved `{mt_path.relative_to(REPO_ROOT)}`"))

rows_ex = []
for t in (30, 45, 60):
    exc_c = f"exceedance_prob_{t}min"
    if exc_c not in merged.columns:
        continue
    e = pd.to_numeric(merged[exc_c], errors="coerce").dropna()
    if len(e) < 5:
        continue
    rows_ex.append(
        {
            "threshold_min": t,
            "exceedance_median": float(e.median()),
            "exceedance_p10": float(e.quantile(0.1)),
            "exceedance_p90": float(e.quantile(0.9)),
            "exceedance_mean": float(e.mean()),
        }
    )
ex_shift = pd.DataFrame(rows_ex)
if not ex_shift.empty:
    ex_path = ART_TAB_PIPE / f"pipeline__05_exceedance_shift_by_threshold__{RID}.csv"
    ex_shift.to_csv(ex_path, index=False)
    display(Markdown(f"Exceedance distribution vs OD threshold → `{ex_path.relative_to(REPO_ROOT)}`"))
    display(ex_shift)
else:
    display(Markdown("_No exceedance columns for 30/45/60 in summary — re-run nb04._"))

mt_df



Saved `artifacts\tables\pipeline\pipeline__05_multithreshold_equity__fit_raw_zscore_x.csv`

Exceedance distribution vs OD threshold → `artifacts\tables\pipeline\pipeline__05_exceedance_shift_by_threshold__fit_raw_zscore_x.csv`

,threshold_min,exceedance_median,exceedance_p10,exceedance_p90,exceedance_mean
0,30,0.000000,0.0,0.004791,0.088017
1,45,0.000031,0.0,0.999969,0.250099
2,60,0.725453,0.0,1.000000,0.513741


,threshold_min,metric,spearman_rho,p_value
0,30,deterministic_jobs_vs_disadvantage_z,0.483806,1.626842e-43
1,45,deterministic_jobs_vs_disadvantage_z,0.467201,2.558810e-40
2,60,deterministic_jobs_vs_disadvantage_z,0.414303,3.110490e-31
3,30,posterior_exceedance_vs_disadvantage_z,-0.301420,1.374617e-16
4,45,posterior_exceedance_vs_disadvantage_z,-0.466943,2.860300e-40
5,60,posterior_exceedance_vs_disadvantage_z,-0.480877,6.138686e-43


## 10. Hook candidates — near Q25 with divergent uncertainty

Because **rank_det ≈ rank_bayes**, the narrative hook uses **threshold uncertainty**: tracts **close to the jobs Q25 cut** with **wide** CI and/or **exceedance** near 0.5. A second export restricts to **exceedance ∈ [0.1, 0.9]** (material ambiguity) among those near Q25.



In [15]:
# Hook: near Q25 jobs cut + high uncertainty / ambiguous exceedance
# Rank stability (ρ ≈ 1) → emphasize *uncertainty* and *tail probability* in prose.
q25_jobs = float(pd.to_numeric(merged[JOB_COL], errors="coerce").quantile(0.25))
tol = 0.05 * max(q25_jobs, 1.0)
near_mask = (merged["posterior_mean_jobs"] - q25_jobs).abs() <= tol
hook_df = merged.loc[near_mask].copy()
if hook_df.empty:
    tol = 0.10 * max(q25_jobs, 1.0)
    near_mask = (merged["posterior_mean_jobs"] - q25_jobs).abs() <= tol
    hook_df = merged.loc[near_mask].copy()
# Score: wide CI × how close exceedance is to 0.5 (decision ambiguity)
ex = hook_df[exc_col].clip(1e-6, 1 - 1e-6)
hook_df["hook_score"] = hook_df["ci_width_log1p"] * (1 - (ex - 0.5).abs() * 2)
hook_df = hook_df.sort_values("hook_score", ascending=False)
cols_h = ["GEOID", "posterior_mean_jobs", JOB_COL, exc_col, "ci_width_log1p", "disadvantage_z", "hook_score"]
cols_h = [c for c in cols_h if c in hook_df.columns]
hook_out = hook_df.head(20)[cols_h]
hook_csv = ART_TAB_PIPE / f"pipeline__05_uncertainty_hook_candidates__{RID}.csv"
hook_out.to_csv(hook_csv, index=False)
display(
    Markdown(
        f"**Q25 jobs** ({JOB_COL}) ≈ {q25_jobs:,.0f}; tolerance ±{tol:,.0f}. "
        f"Candidates: **{len(hook_df)}** tracts. Exported top 20 by hook_score."
    )
)
display(Markdown(f"Saved `{hook_csv.relative_to(REPO_ROOT)}`"))

# Near Q25 on posterior mean *and* exceedance in (0.1, 0.9) — uncertainty hook when ranks barely move
ambig_mask = (hook_df[exc_col] >= 0.1) & (hook_df[exc_col] <= 0.9)
ambig_df = hook_df.loc[ambig_mask].sort_values("ci_width_log1p", ascending=False)
ambig_csv = ART_TAB_PIPE / f"pipeline__05_hook_near_q25_exceedance_ambiguous__{RID}.csv"
ambig_out = ambig_df.head(15)[cols_h] if not ambig_df.empty else ambig_df
ambig_out.to_csv(ambig_csv, index=False)
display(
    Markdown(
        f"**Near-Q25 + ambiguous exceedance (0.1–0.9):** **{len(ambig_df)}** tracts; top 15 by CI width → `{ambig_csv.relative_to(REPO_ROOT)}`"
    )
)
hook_out



**Q25 jobs** (jobs_C000_45min) ≈ 4,470; tolerance ±224. Candidates: **21** tracts. Exported top 20 by hook_score.

Saved `artifacts\tables\pipeline\pipeline__05_uncertainty_hook_candidates__fit_raw_zscore_x.csv`

**Near-Q25 + ambiguous exceedance (0.1–0.9):** **21** tracts; top 15 by CI width → `artifacts\tables\pipeline\pipeline__05_hook_near_q25_exceedance_ambiguous__fit_raw_zscore_x.csv`

,GEOID,posterior_mean_jobs,jobs_C000_45min,exceedance_prob_45min,ci_width_log1p,disadvantage_z,hook_score
344,06073013317,4470.527929,4467.0,0.499781,0.341225,-0.475744,0.341076
387,06073013908,4492.789134,4457.0,0.483656,0.348215,0.251205,0.336832
501,06073017057,4492.025882,4483.0,0.475844,0.342192,-0.730586,0.325659
478,06073017031,4490.498913,4471.0,0.477125,0.341055,-0.663490,0.325452
337,06073013306,4428.595359,4402.0,0.567000,0.336153,0.326796,0.291108
268,06073009902,4531.351847,4659.0,0.370750,0.384175,0.000000,0.284866
453,06073016619,4388.471858,4378.0,0.613031,0.351551,0.282248,0.272078
489,06073017045,4550.545313,4551.0,0.392437,0.341586,-1.338082,0.268102
488,06073017044,4383.899555,4371.0,0.616469,0.336642,-1.056016,0.258225
528,06073017306,4561.418393,4575.0,0.375219,0.338062,-1.414054,0.253694
